<a href="https://colab.research.google.com/github/Nourhasann/simple-rag-chatbot/blob/main/Chatbot_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install openai numpy -q

In [49]:
import os
import numpy as np
from openai import OpenAI


client = OpenAI()  # reads OPENAI_API_KEY from environment (kept here in case your quota is restored later)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"
TOP_K = 3  # how many similar QA pairs to retrieve as context

## Step 1: Load the dataset


In [44]:
import pandas as pd

DATA_PATH = "Conversation.csv"

def load_dataset(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=["question", "answer"])
    questions = df["question"].astype(str).tolist()
    answers = df["answer"].astype(str).tolist()
    return questions, answers

questions, answers = load_dataset(DATA_PATH)
print(f"Loaded {len(questions)} QA pairs")
print(questions[:3])
print(answers[:3])

Loaded 3725 QA pairs
['hi, how are you doing?', "i'm fine. how about yourself?", "i'm pretty good. thanks for asking."]
["i'm fine. how about yourself?", "i'm pretty good. thanks for asking.", 'no problem. so how have you been?']


#Step 2: Tokenization


In [45]:
!pip install tiktoken -q

import tiktoken

encoding = tiktoken.encoding_for_model(CHAT_MODEL)
sample_text = questions[0]
tokens = encoding.encode(sample_text)

print(f"Text: {sample_text}")
print(f"Token IDs: {tokens}")
print(f"Number of tokens: {len(tokens)}")

Text: hi, how are you doing?
Token IDs: [3686, 11, 1495, 553, 481, 5306, 30]
Number of tokens: 7


## Step 3: Embedding


In [50]:
!pip install sentence-transformers -q

In [51]:
from sentence_transformers import SentenceTransformer

print("Loading sentence-transformer model...")
# Load a pre-trained model
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully.")

print("Generating embeddings for the dataset using sentence-transformers...")
# Generate embeddings
question_embeddings_free = sentence_model.encode(questions, show_progress_bar=True)
print(f"Embeddings shape: {question_embeddings_free.shape}")

Loading sentence-transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully.
Generating embeddings for the dataset using sentence-transformers...


Batches:   0%|          | 0/117 [00:00<?, ?it/s]

Embeddings shape: (3725, 384)


## Step 5: Retrieve relevant context


When the user asks something, embed their message with the **same** `sentence_model` used for the dataset (embeddings from different models aren't comparable), then compare it to every stored question using cosine similarity. Pull out the top matches as context.

In [52]:
def cosine_similarity(a, b):
    a_norm = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = b / np.linalg.norm(b)
    return a_norm @ b_norm

def get_relevant_context(user_input, k=TOP_K):
    query_embedding = sentence_model.encode([user_input])[0]
    sims = cosine_similarity(question_embeddings_free, query_embedding)
    top_indices = np.argsort(sims)[-k:][::-1]

    context_pairs = []
    for idx in top_indices:
        context_pairs.append(f"Q: {questions[idx]}\nA: {answers[idx]}")
    return "\n\n".join(context_pairs)

# quick test
test_context = get_relevant_context("how are you doing today")
print(test_context)

Q: how are you doing today?
A: i'm doing great. what about you?

Q: hi, how are you doing?
A: i'm fine. how about yourself?

Q: i'm doing well. how about you?
A: never better, thanks.


## Step 6: Generate a response


In [53]:
!pip install huggingface_hub -q

In [74]:
from huggingface_hub import InferenceClient
from google.colab import userdata


# IMPORTANT: Ensure your Hugging Face API token is set as a Colab Secret named 'HF_TOKEN'.
# You can add secrets using the 🔑 icon in the left sidebar.
HF_TOKEN = userdata.get('HF_TOKEN')
hf_client = InferenceClient(token=HF_TOKEN)
# Using a commonly available free model that should work with a valid token.
HF_CHAT_MODEL = "Qwen/Qwen2.5-7B-Instruct"
def ask_chatbot(user_input, context, history):
    system_prompt = (
        "You are a helpful chatbot. Use the example Q&A pairs below as a style "
        "and knowledge reference, but respond naturally to the user's actual question.\n\n"
        f"Reference examples:\n{context}"
    )

    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(history)
    messages.append({"role": "user", "content": user_input})

    response = hf_client.chat.completions.create(
        model=HF_CHAT_MODEL,
        messages=messages,
        max_tokens=300,
    )
    return response.choices[0].message.content

# quick test
test_reply = ask_chatbot("how are you doing today", test_context, [])
print(test_reply)

I'm doing great, thanks for asking! How about you?


# Step 7: Keep conversation history

In [75]:
history = []

def chat(user_input):
    global history
    context = get_relevant_context(user_input)
    reply = ask_chatbot(user_input, context, history)

    history.append({"role": "user", "content": user_input})
    history.append({"role": "assistant", "content": reply})

    if len(history) > 20:
        history = history[-20:]

    return reply

#Test

In [76]:
print(chat("Hi, what's your name?"))

Hello! You can call me Chatbot. How about you?


In [77]:
print(chat("What do you like to do for fun?"))

I enjoy helping with fun conversations and answering questions! What about you—do you have any hobbies or activities you really enjoy?


# interactive loop

In [78]:
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ("quit", "exit"):
        break
    if not user_input:
        continue
    print(f"Bot: {chat(user_input)}\n")

You: hi
Bot: Hi there! How can I assist you today?

You: whats your name
Bot: You can call me Chatbot. How about you? What's your name?

You: nour
Bot: Nice to meet you, Nour! How can I help you today?

You: tell me whats chatpbot providere
Bot: Chatbot provides a range of services like answering questions, having conversations, and offering information. Is there a specific aspect of chatbot services you're interested in learning more about?

You: when building chatbots i mean
Bot: When building chatbots, you typically focus on creating conversational interfaces that can understand and respond to user inputs. This involves several key steps:

1. **Define the Purpose**: Determine what the chatbot will do and who it will interact with.
2. **Design the Conversation Flow**: Map out how the chatbot will handle different types of user inputs and provide appropriate responses.
3. **Choose a Platform**: Select a platform or framework to build your chatbot, such as Dialogflow, Microsoft Bot Fra

KeyboardInterrupt: Interrupted by user

In [ ]:
!pip install gradio -q


In [79]:

import gradio as gr

def gradio_chat(user_input, gr_history):
    reply = chat(user_input)  # reuses your existing chat() function and its own history tracking
    return reply

demo = gr.ChatInterface(
    fn=gradio_chat,
    title="My Chatbot",
    description="Ask me anything — powered by Qwen2.5 via Hugging Face.",
    examples=["Hi, what's your name?", "How are you doing today?", "What do you like to do for fun?"],
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://74b49fe3cdb6d306a2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
